# 1. Évolution de l'architecture : Du Perceptron au CNN

Notre objectif est de classifier des panneaux de signalisation de 48x48 pixels en 43 classes (Dataset GTSRB). 
Nous avons d'abord conçu un modèle de référence (Baseline - MLP) pour valider notre pipeline de données, avant de construire un Réseau de Neurones Convolutif (CNN) capable de comprendre la géométrie des panneaux.

In [1]:
from trafficsigns.models.baseline import build_baseline_model
from trafficsigns.models.cnn import build_cnn_model

input_shape = (48, 48, 3)
num_classes = 43

## 1.1 Le Modèle de Référence (MLP)
Le modèle de base aplatit directement l'image (Flatten) en un vecteur de 6912 pixels. L'inconvénient est qu'il perd toute notion de géométrie et de contexte spatial.

In [2]:
baseline_model = build_baseline_model(input_shape, num_classes)
baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 6912)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       884,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 890,411 (3.40 MB)

 Trainable params: 890,411 (3.40 MB)

 Non-trainable params: 0 (0.00 B)

## 1.2 Le Réseau de Neurones Convolutif (CNN)
Pour pallier ce problème, nous avons conçu une architecture CNN inspirée de VGG :
- **Conv2D (3x3)** : extrait les caractéristiques locales (bords, formes, chiffres).
- **MaxPooling2D (2x2)** : divise mathématiquement la taille par 2 (48->24->12->6) pour synthétiser l'information.
- **Dropout (0.5)** : désactive 50% des neurones pour forcer le modèle à généraliser et éviter l'overfitting.

In [3]:
cnn_model = build_cnn_model(input_shape, num_classes)
cnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 43)             │        11,051 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,284,203 (4.90 MB)

 Trainable params: 1,284,203 (4.90 MB)

 Non-trainable params: 0 (0.00 B)